<a href="https://colab.research.google.com/github/rafaalathais/analise-gasto-parlamentar/blob/main/notebook_gasto_parlamentar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Análise exploratória dos gastos da Cota para o Exercício da Atividade Parlamentar dos Senadores Brasileiros.
**Disciplina**: IMD1151 — Ciências de Dados

**Grupo**:
*   Gabriel Ferreira Cavalcante
*   Jordson Albino Galvão Tavares
*   Rafaela Thais de Abreu Lúcio Maia

## Introdução

Este projeto tem como objetivo realizar uma Análise Exploratória de Dados sobre os gastos da Cota para o Exercício da Atividade Parlamentar dos Senadores (CEAPS), buscando identificar padrões, tendências e diferenças na utilização dos recursos públicos destinados à atividade parlamentar.

A análise pretende contemplar dados históricos de 2008 a 2026'.

## Questões norteadoras

O projeto busca responder às seguintes perguntas:

- Em quais períodos do ano os senadores apresentam maiores gastos? Como essas despesas evoluíram ao longo dos anos?

- Quais unidades federativas apresentam os maiores gastos associados aos seus senadores?

- Quais partidos apresentam os maiores gastos totais e médios, tanto nacionalmente quanto por unidade federativa?

- Quais senadores apresentam os maiores gastos da cota parlamentar?

- Quais tipos de despesas representam a maior parcela dos gastos parlamentares?

Como possível extensão do projeto, pretende-se investigar a identificação de perfis de senadores com base em seus padrões de despesas.

## Fonte dos dados

Os dados serão obtidos por meio do Portal de Dados Abertos do Senado Federal.

## Objetivo

Serão realizadas a obtenção dos registros históricos, a identificação das variáveis disponíveis e a verificação inicial de problemas como dados ausentes, inconsistências e diferenças de estrutura entre os arquivos.

Os dados consolidados servirão de base para as etapas posteriores de limpeza, transformação e análise exploratória.

## 1. Coleta de dados

In [ ]:
import time
from http.client import IncompleteRead
from io import BytesIO
import unicodedata

import pandas as pd
import requests
from IPython.display import display

url_base = 'https://adm.senado.gov.br/adm-dadosabertos/api/v1/senadores/despesas_ceaps/{}/csv'
anos = range(2008, 2027)
max_tentativas = 3

In [ ]:
if 'dfs_despesas' not in globals():
    dfs_despesas = {}

anos_falha = []

for ano in anos:
    if ano in dfs_despesas:
        print(f'{ano}: reutilizado ({len(dfs_despesas[ano])} registros já carregados)')
        continue

    url = url_base.format(ano)
    for tentativa in range(1, max_tentativas + 1):
        try:
            resposta = requests.get(url, timeout=(15, 120))
            resposta.raise_for_status()
            df_ano = pd.read_csv(BytesIO(resposta.content), sep=';', decimal=',')
            dfs_despesas[ano] = df_ano
            print(f'{ano}: {len(df_ano)} registros coletados')
            break
        except (requests.exceptions.RequestException, IncompleteRead, OSError) as erro:
            if tentativa < max_tentativas:
                print(f'{ano}: tentativa {tentativa} falhou ({erro}); nova tentativa em {2 ** tentativa} s')
                time.sleep(2 ** tentativa)
            else:
                print(f'{ano}: falha após {max_tentativas} tentativas: {erro}')
                anos_falha.append(ano)

2008: reutilizado (2449 registros já carregados)
2009: reutilizado (16480 registros já carregados)
2010: reutilizado (18235 registros já carregados)
2011: reutilizado (25574 registros já carregados)
2012: reutilizado (30175 registros já carregados)
2013: reutilizado (29376 registros já carregados)
2014: reutilizado (22002 registros já carregados)
2015: reutilizado (26055 registros já carregados)
2016: reutilizado (26768 registros já carregados)
2017: reutilizado (27023 registros já carregados)
2018: reutilizado (24492 registros já carregados)
2019: reutilizado (21634 registros já carregados)
2020: reutilizado (14088 registros já carregados)
2021: reutilizado (16827 registros já carregados)
2022: reutilizado (16804 registros já carregados)
2023: reutilizado (18822 registros já carregados)
2024: reutilizado (21431 registros já carregados)
2025: reutilizado (23819 registros já carregados)
2026: 15216 registros coletados


In [ ]:
# Visualizar os dados de 2025
dfs_despesas[2010].head(5)

,ID,TIPO_DOCUMENTO,ANO,MÊS,COD_SENADOR,NOME_SENADOR,TIPO_DESPESA,CPF_CNPJ_FORNECEDOR,NOME_FORNECEDOR,DOCUMENTO,DATA,DETALHAMENTO,VALOR_REEMBOLSADO
0,288070,Recibo,2010,7,17,EDUARDO SUPLICY,"Locomoção, hospedagem, alimentação, combustíve...",02.555.926/0001-79,Centrovias Sistemas Rodoviários S/,000529,2010-07-16,NaN,7.25
1,288078,Recibo,2010,7,17,EDUARDO SUPLICY,"Locomoção, hospedagem, alimentação, combustíve...",02.555.926/0001-79,Centrovias Sistemas Rodoviários S/A,000560,2010-07-17,NaN,12.05
2,288058,Recibo,2010,7,17,EDUARDO SUPLICY,"Locomoção, hospedagem, alimentação, combustíve...",02.555.926/0001-79,Centrovias Sistemas Rodoviários S/A,000529,2010-07-30,NaN,7.25
3,288062,Recibo,2010,7,17,EDUARDO SUPLICY,"Locomoção, hospedagem, alimentação, combustíve...",02.555.926/0001-79,Centrovias Sistemas Rodoviários S/A,000543,2010-07-30,NaN,12.05
4,288069,Recibo,2010,7,17,EDUARDO SUPLICY,"Locomoção, hospedagem, alimentação, combustíve...",02.451.848/0001-62,Conc. Sistema Anhanguera Bandeirantes SA,136156,2010-07-16,NaN,4.25


In [ ]:
anos_obtidos = sorted(dfs_despesas)
anos_faltantes = sorted(set(anos) - set(anos_obtidos))

if dfs_despesas:
    colunas_referencia = list(dfs_despesas[anos_obtidos[0]].columns)
    estrutura_arquivos = []
    for ano, quadro in sorted(dfs_despesas.items()):
        colunas = list(quadro.columns)
        estrutura_arquivos.append({
            'ano_arquivo': ano,
            'quantidade_colunas': len(colunas),
            'colunas_ausentes_vs_referencia': ', '.join(c for c in colunas_referencia if c not in colunas),
            'colunas_adicionais_vs_referencia': ', '.join(c for c in colunas if c not in colunas_referencia),
        })
    tabela_estrutura = pd.DataFrame(estrutura_arquivos)
    despesas_por_ano = pd.concat(
        [quadro.assign(ANO_ARQUIVO=ano) for ano, quadro in sorted(dfs_despesas.items())],
        ignore_index=True, sort=False
    )
else:
    colunas_referencia = []
    tabela_estrutura = pd.DataFrame(columns=[
        'ano_arquivo', 'quantidade_colunas',
        'colunas_ausentes_vs_referencia', 'colunas_adicionais_vs_referencia'
    ])
    despesas_por_ano = pd.DataFrame()

despesas_brutas = despesas_por_ano.copy()
total_registros_anuais = sum(len(quadro) for quadro in dfs_despesas.values())
print(f'Anos obtidos ({len(anos_obtidos)}): {anos_obtidos}')
print(f'Anos com falha ou ainda não obtidos ({len(anos_faltantes)}): {anos_faltantes}')
print(f'Coleta completa de 2008 a 2025: {len(anos_faltantes) == 0}')
print(f'Registros na soma anual: {total_registros_anuais:,}')
print(f'Registros na base consolidada: {len(despesas_brutas):,}')
assert total_registros_anuais == len(despesas_brutas)

display(tabela_estrutura)

Anos obtidos (18): [2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Anos com falha ou ainda não obtidos (0): []
Coleta completa de 2008 a 2025: True
Registros na soma anual: 382,042
Registros na base consolidada: 382,042


,ano_arquivo,quantidade_colunas,colunas_ausentes_vs_referencia,colunas_adicionais_vs_referencia
0,2008,13,,
1,2009,13,,
2,2010,13,,
3,2011,13,,
4,2012,13,,
5,2013,13,,
6,2014,13,,
7,2015,13,,
8,2016,13,,
9,2017,13,,


## 2. Dicionário de dados
A tabela abaixo lista todas as colunas da base, seu tipo e a decisão de mantê-los ou não na análise exploratória. O percentual de nulos refere-se há quantidade total de valores ausentes na base de dados entre os anos de 2008 até 2025.

| Coluna | Tipo | Nulos (%) | Decisão | Justificativa
|--------|------|-----------| -----|-----|
|**`ID`** | int |0% |**Manter**| Identificador único da despesa - útil para evitar duplicação
|**`TIPO_DOCUMENTO`** |texto|0.8% |Descartar | Tipo do comprovante da despesa (ex:'Boleto', 'Nota fiscal') - pouco valor analítico
|**`ANO`** | int|0%|**Manter** |Valor único (ex: 2025) constante e sem variância - necessário para a análise temporal
|**`MÊS`**| int|0%|**Manter** | Permite identificar os meses com maiores gastos
|**`COD_SENADOR`**| int|0% |**Manter** |Identificador único do parlamentar
|**`NOME_SENADOR`**| texto|0% |**Manter** |Essencial para legibilidade
|**`TIPO_DESPESA`** |texto|0,1%  |**Manter** | Tipo da despesa (ex: 'Locomoção, hospedagem, alimentação, combustíveis e lubrificantes', 'Divulgação da atividade parlamentar')
|**`CPF_CNPJ_FORNECEDOR`**|texto|0,8% |**Manter** | Identificador único do fornecedor - evitar duplicações/erros de digitação do nome do fornecedor
|**`NOME_FORNECEDOR`** |texto |0,8% |**Manter**|Essencial para legibilidade
|**`DOCUMENTO`** |texto |7,2% |Descartar |Número sequencial ou texto livre
|**`DATA`** |texto |0.8% |Descartar | Data em que foi emitido o comprovante (redundante com `ANO` e `MÊS`)
|**`DETALHAMENTO`** |texto|45,8% |**Manter** | Indica informações da despesa
|**`VALOR_REEMBOLSADO`**|float|0%|**Manter** |Valor exato que foi devolvido ao parlamentar - métrica central

In [ ]:
total_valores_unicos = 0
total_nulos = 0
total_linhas = 0

for ano in anos:
  total_nulos += dfs_despesas[ano].isnull().sum()

  total_valores_unicos += dfs_despesas[ano].nunique()
  total_linhas += len(dfs_despesas[ano])

total_nulos_pct = (total_nulos / total_linhas * 100).round(1)

resumo_nulos = pd.DataFrame({
      'valores únicos': total_valores_unicos,
      'nulos': total_nulos,
      'nulos (%)': total_nulos_pct
  }).sort_values('nulos (%)', ascending=False)

resumo_nulos

,valores únicos,nulos,nulos (%)
DETALHAMENTO,129031,175146,45.8
DOCUMENTO,313631,27534,7.2
NOME_FORNECEDOR,89818,3144,0.8
CPF_CNPJ_FORNECEDOR,64354,3144,0.8
TIPO_DOCUMENTO,111,3144,0.8
DATA,7664,3144,0.8
TIPO_DESPESA,120,284,0.1
ANO,18,0,0.0
ID,382042,0,0.0
COD_SENADOR,1677,0,0.0


## 3. Avaliação
Avaliação da viabilidade e da qualidade dos dados obtidos:

Nossa avaliação é de que o projeto é totalmente viável. Como o Senado disponibiliza essas informações de forma pública e estruturada, conseguimos baixar com sucesso todo o histórico de despesas de 2008 a 2025, consolidando uma base robusta de 382.042 registros.

Ao avaliar a qualidade dessa base, encontramos:

- Boa padronização, o Senado manteve exatamente a mesma estrutura de 13 colunas em todos os arquivos, ao longo desses 18 anos. Isso nos garantiu a concatenação dos dados de forma direta.
- A maioria das colunas estão preenchidas, colunas vitais para nossa análise (Como: ANO, MÊS, VALOR_REEMBOLSADO) estão 100% preenchidas.
- A coluna DETALHAMENTO não é confiavel, está vazia em quase metade da base, 45.8% de valores nulos. Em virtude disso, sabemos que não poderemos depender dela para analises muito profundas sobre a natureza das despesas, focando na coluna TIPO_DESPESA. Porém, podemos aproveitar esse campo para fazer, justamente, análises sobre a nulidade dos dados.


## 4. Conclusão
Principais conclusões e aprendizados até o momento:

Como a estrutura das colunas se manteve a mesma, concluimos que para o processo de limpeza, precisaremos focar em resolver inconsistencias de texto (nomes de fornecedores e senadores com digitações e/ou acentuações diferentes ao longo dos anos - se houver).

As próximas atividades são:
- Subir código-fone em repositório git (Gabriel)
- Limpeza dos dados das CEAPS (Gabriel)
- Coleta dos dados de senadores e ex-senadores (Rafa)
- Limpeza e normalização dos dados de ex-senadores (Gabriel)
- Planejamento de tópicos da Análise (Rafa)
- Elaborar gráficos de distribuição e primeiras respostas (Jordson)

A API demonstrou instabilidades e falhas, o que demonstrou a necessidade de criar scripts com lógica forte para tratamento de exceções.